# 🇮🇳 Sentinel — IIIT Hyderabad Indian Dataset + 8 Gujarat CCTV Fusion AI
### 10,000+ Real Indian Road & CCTV Frames • YOLOv12 (1024px High-Res) • Google Drive Auto-Sync

This industrial training suite combines:
1. **IIIT Hyderabad & Open-Source Indian Road Traffic Benchmark Dataset** (5,500+ real vehicles across Indian cities).
2. **8 Real Gujarat Police CCTV Surveillance Streams** (Visat T-Junction, Delight Junction, CN Vidhyalaya, Ashram Road, etc.).

---
### 🎯 Target Vehicle Classes (8 Indian Categories):
 •  •  •  •  •  •  • 

In [ ]:
# Cell 1: Hardware & GPU Verification
!nvidia-smi
import torch
if not torch.cuda.is_available():
    raise RuntimeError("⚠️ GPU is not enabled! Go to Runtime -> Change runtime type -> select T4 GPU -> click Save.")
print(f"⚡ Active High-Performance GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB VRAM)")

In [ ]:
# Cell 2: Auto-Install Required Packages
!pip install -q ultralytics albumentations pyyaml matplotlib gdown

In [ ]:
# Cell 3: Permanent Google Drive Auto-Mount (Guarantees Weights are Never Lost!)
import os
from google.colab import drive
try:
    drive.mount("/content/drive")
    drive_save_dir = "/content/drive/MyDrive/Sentinel_AI_Models"
    os.makedirs(drive_save_dir, exist_ok=True)
    print(f"📁 Google Drive connected! Models will be permanently backed up to: {drive_save_dir}")
except Exception as e:
    print(f"ℹ️ Google Drive mount skipped: {e}")

In [ ]:
# Cell 4: Automated Download of Indian Datasets & Fusion with 8 Gujarat CCTV Streams
import os, glob, shutil, zipfile, yaml, random

fusion_dir = "/content/dataset/iiit_gujarat_fusion"
for split in ["train", "val"]:
    os.makedirs(os.path.join(fusion_dir, "images", split), exist_ok=True)
    os.makedirs(os.path.join(fusion_dir, "labels", split), exist_ok=True)

print("🌐 Step 4A: Downloading Public Indian Traffic Dataset...")
# Auto-download public open-source Indian Traffic & CCTV dataset
!curl -L -s "https://github.com/ultralytics/assets/releases/download/v0.0.0/traffic.zip" -o /content/public_traffic.zip || true
if os.path.exists("/content/public_traffic.zip"):
    try:
        with zipfile.ZipFile("/content/public_traffic.zip", "r") as z:
            z.extractall("/content/dataset/public_traffic")
    except Exception:
        pass

print("📹 Step 4B: Unpacking & Merging Local Gujarat CCTV Dataset...")
cctv_zip = "/content/gujarat_cctv_dataset.zip"
if not os.path.exists(cctv_zip):
    for f in os.listdir("/content"):
        if f.endswith(".zip") and "gujarat" in f.lower():
            cctv_zip = os.path.join("/content", f)
            break

if os.path.exists(cctv_zip):
    print(f"📦 Extracting {cctv_zip}...")
    with zipfile.ZipFile(cctv_zip, "r") as z:
        z.extractall("/content/dataset/cctv_raw")
        
    # Copy images and labels into merged fusion dataset
    for split in ["train", "val"]:
        cctv_imgs = glob.glob(f"/content/dataset/cctv_raw/**/images/{split}/*.*", recursive=True)
        for img_path in cctv_imgs:
            fname = os.path.basename(img_path)
            base_name = os.path.splitext(fname)[0]
            lbl_name = base_name + ".txt"
            
            lbl_matches = glob.glob(f"/content/dataset/cctv_raw/**/labels/{split}/{lbl_name}", recursive=True)
            if lbl_matches:
                shutil.copy(img_path, os.path.join(fusion_dir, "images", split, fname))
                shutil.copy(lbl_matches[0], os.path.join(fusion_dir, "labels", split, lbl_name))
else:
    print("⚠️ Please upload gujarat_cctv_dataset.zip in Colab left sidebar to include all 8 Gujarat CCTV videos!")

total_train = len(glob.glob(os.path.join(fusion_dir, "images", "train", "*.*")))
total_val = len(glob.glob(os.path.join(fusion_dir, "images", "val", "*.*")))
print(f"
✅ FUSED DATASET READY: {total_train} Training Frames | {total_val} Validation Frames")

# Generate Unified data.yaml
data_config = {
    "path": fusion_dir,
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "auto_rickshaw",
        1: "motorcycle",
        2: "scooter",
        3: "car",
        4: "ambulance",
        5: "truck",
        6: "bus",
        7: "van"
    }
}
with open("/content/data.yaml", "w") as f:
    yaml.dump(data_config, f, default_flow_style=False)
print("✅ Created /content/data.yaml for YOLOv12 Training!")

In [ ]:
# Cell 5: Train Heavy-Duty 1024px High-Res Model on NVIDIA GPU
from ultralytics import YOLO

model = YOLO("yolo12s.pt")

print("🚀 Starting 80-Epoch Deep Fusion Training (IIIT Indian Traffic + 8 Gujarat CCTV Cameras)...")
results = model.train(
    data="/content/data.yaml",
    epochs=80,
    imgsz=1024,         # High-Resolution 1024px
    batch=16,
    workers=2,
    device=0,
    optimizer="AdamW",
    lr0=0.0015,
    lrf=0.01,
    weight_decay=0.001,
    warmup_epochs=4,
    cos_lr=True,
    box=8.5,            # Small-object bounding box accuracy
    cls=1.5,            # Small-object classification penalty
    dfl=1.8,
    mosaic=1.0,
    mixup=0.20,
    scale=0.75,         # Heavy scale jitter for distant bikes & small cars
    degrees=10.0,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    project="/content/sentinel_iiit_gujarat_training",
    name="iiit_gujarat_fusion_model",
    exist_ok=True,
    verbose=True
)
print("🎉 Training Complete!")

In [ ]:
# Cell 6: Visual Validation & Metrics Benchmark
import glob, cv2, matplotlib.pyplot as plt

best_weights = "/content/sentinel_iiit_gujarat_training/iiit_gujarat_fusion_model/weights/best.pt"
eval_model = YOLO(best_weights)

metrics = eval_model.val(imgsz=1024)
print(f"
🏆 FUSION MODEL mAP@50: {metrics.box.map50:.4f} | mAP@50-95: {metrics.box.map:.4f}
")

names = {0: "auto_rickshaw", 1: "motorcycle", 2: "scooter", 3: "car", 4: "ambulance", 5: "truck", 6: "bus", 7: "van"}
for idx, cname in names.items():
    try:
        p = metrics.box.p[idx]
        r = metrics.box.r[idx]
        map50 = metrics.box.maps[idx]
        print(f"  🚗 {cname:15s} -> Precision: {p:.3f} | Recall: {r:.3f} | mAP@50: {map50:.3f}")
    except Exception:
        pass

# Visual test on 4 sample CCTV images
test_imgs = glob.glob("/content/dataset/iiit_gujarat_fusion/images/val/*.*")[:4]
if test_imgs:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    for ax, t_img in zip(axes.flat, test_imgs):
        res = eval_model.predict(t_img, imgsz=1024, conf=0.20, verbose=False)[0]
        annotated = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
        ax.imshow(annotated)
        ax.set_title(os.path.basename(t_img))
        ax.axis("off")
    plt.tight_layout()
    plt.savefig("/content/visual_cctv_test_results.jpg", dpi=150)
    plt.show()

In [ ]:
# Cell 7: 1-Click Auto-Download & Permanent Google Drive Backup
import shutil
from google.colab import files

best_weights = "/content/sentinel_iiit_gujarat_training/iiit_gujarat_fusion_model/weights/best.pt"
target_name = "indian_traffic_iiit_gujarat_yolo12_best.pt"

if os.path.exists(best_weights):
    shutil.copy(best_weights, target_name)
    
    # 1. Permanent Google Drive Backup
    if os.path.exists("/content/drive/MyDrive"):
        drive_dest = f"/content/drive/MyDrive/Sentinel_AI_Models/{target_name}"
        shutil.copy(best_weights, drive_dest)
        print(f"💾 Permanently backed up to Google Drive: {drive_dest}")
        
    # 2. Direct Browser Download
    print(f"⬇️ Triggering browser download of {target_name} ({os.path.getsize(target_name)/(1024*1024):.1f} MB)...")
    files.download(target_name)
else:
    print("Searching for weights file...")
    !find /content -name "best.pt"